# Advanced Anomaly Detection Pipeline v3 — Triple Training Data
## Optimized for F1 Score under Class Imbalance

**What changed from v2:**
- **3× training batches** (3060 users, 260 anomalous) — `training_batch`, `first_batch`, `second_batch`
- **127 features** (up from 86) — NMF embeddings, rating transitions, popularity-quartile behavior, item overlap, deviation asymmetry
- **Multi-seed CV** — 2 independent CV seeds averaged for variance reduction
- **4 LightGBM + 3 XGBoost configs** — more diversity in strongest model families
- **ExtraTrees added** — complementary tree ensemble with random splits
- **Stacking with multiple C values** — more robust meta-learner
- **10-fold CV** — 260 positives makes per-fold estimates very stable (~26 per fold)


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os
# Update this path to your working directory
os.chdir('/content/drive/My Drive/cs421 principles of machine learning/group project/leong/iter 6')
print('Working directory:', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import warnings, json, itertools
warnings.filterwarnings("ignore")

from scipy import stats
from scipy.sparse import csr_matrix
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, classification_report)
from sklearn.ensemble import (IsolationForest, RandomForestClassifier,
                              GradientBoostingClassifier, ExtraTreesClassifier)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb

print("All imports successful.")

## 1. Load and Combine All Three Training Batches

Three labeled batches combined:
- `training_batch_with_labels.npz` — users 100–1199 (1100 users, 100 anomalous)
- `first_batch_with_labels.npz` — users 2500–3599 (1100 users, 100 anomalous)
- `second_batch_with_labels.npz` — users 4100–4959 (860 users, 60 anomalous)

Combined: **3060 users (2800 normal, 260 anomalous)**.

Test: `third_batch.npz` — users 5500–7124 (1625 users, no labels).


In [ ]:
# Load all three training batches and test data
train_data_1 = np.load("training_batch_with_labels.npz")
train_data_2 = np.load("first_batch_with_labels.npz")
train_data_3 = np.load("second_batch_with_labels.npz")
test_data    = np.load("third_batch.npz")

# Combine all three training batches
train_ratings = pd.DataFrame(
    np.concatenate([train_data_1["X"], train_data_2["X"], train_data_3["X"]]),
    columns=["user", "item", "rating"]
)
labels = pd.DataFrame(
    np.concatenate([train_data_1["y"], train_data_2["y"], train_data_3["y"]]),
    columns=["user", "label"]
).set_index("user")

test_ratings = pd.DataFrame(test_data["X"], columns=["user", "item", "rating"])

# Combined ratings for computing global item statistics
all_ratings = pd.concat([train_ratings, test_ratings], ignore_index=True)
n_items = 1000

print(f"Training: {train_ratings['user'].nunique()} users, {len(train_ratings)} interactions")
print(f"Test:     {test_ratings['user'].nunique()} users, {len(test_ratings)} interactions")
print(f"Items:    {all_ratings['item'].nunique()}")
print(f"Rating range: {all_ratings['rating'].min()} - {all_ratings['rating'].max()}")
print(f"\nLabel distribution:")
print(labels['label'].value_counts().rename({0: 'Normal', 1: 'Anomalous'}))
print(f"\nAnomaly rate: {labels['label'].mean():.1%}")

## 2. Feature Engineering (127 features across 11 categories)

| Category | Features | Count |
|----------|----------|-------|
| A. Basic Stats | mean, std, min, max, median, count, sum, range | 8 |
| B. Distribution Shape | skew, kurt, entropy, MAD, IQR, CV, per-rating proportions, transitions, runs, percentiles | 27 |
| C. Interaction Structure | unique items, repeat ratio, density, repeat stats | 5 |
| D. Item-Normalized Deviation | residuals, z-scores, large deviations, asymmetry | 13 |
| E. Popularity-Aware | weighted ratings, item pop stats, popularity-quartile means | 8 |
| F. Sequence Patterns | drift, trend, autocorrelation, segment features | 5 |
| G. Item Diversity | item entropy, Gini-Simpson, unique ratio | 3 |
| H. SVD Latent | 35 SVD components + error + norm | 37 |
| H2. NMF Latent | 15 NMF components + error + norm | 17 |
| I. Cosine Similarity | to average user profile | 1 |
| J. Rating-Pop Correlation | Pearson correlation of user ratings vs item popularity | 1 |
| K. Item Overlap | overlap between first/second half of rated items | 1 |
| + Unsupervised | 2 Isolation Forest + 1 LOF anomaly scores | 3 |

In [ ]:
# Precompute global item statistics from ALL data (train + test)
g_item_stats = all_ratings.groupby("item")["rating"].agg(["mean", "std", "count"])
g_item_stats.columns = ["item_mean", "item_std", "item_count"]
g_item_stats["item_std"] = g_item_stats["item_std"].fillna(0)

g_item_pop = all_ratings.groupby("item")["rating"].count()
g_mean = all_ratings["rating"].mean()

print(f"Global mean rating: {g_mean:.3f}")
print(f"Items with stats: {len(g_item_stats)}")

In [ ]:
def compute_features(df, svd_model=None, nmf_model=None, fit=False):
    """Compute comprehensive user-level features from interaction data."""
    feats = []
    users_list = sorted(df["user"].unique())

    # ===== A. Basic Rating Statistics =====
    basic = df.groupby("user")["rating"].agg(
        mean_r="mean", std_r="std", min_r="min", max_r="max",
        med_r="median", n_ratings="count", sum_r="sum"
    ).fillna(0)
    basic["range_r"] = basic["max_r"] - basic["min_r"]
    feats.append(basic)

    # ===== B. Distribution Shape (expanded) =====
    def dist_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        d["skew"] = stats.skew(r) if n >= 3 else 0
        d["kurt"] = stats.kurtosis(r) if n >= 4 else 0
        vc = pd.Series(r).value_counts(normalize=True)
        d["entropy"] = stats.entropy(vc)
        d["mode_prop"] = vc.iloc[0]
        d["mad"] = np.median(np.abs(r - np.median(r)))
        d["iqr"] = np.percentile(r, 75) - np.percentile(r, 25) if n >= 4 else 0
        d["cv"] = (np.std(r) / np.mean(r)) if np.mean(r) > 0 else 0
        for rv in range(6): d[f"pr{rv}"] = np.mean(r == rv)
        d["p_extreme"] = np.mean((r == 0) | (r == 5))
        d["p_low"] = np.mean(r <= 1)
        d["p_high"] = np.mean(r >= 4)
        d["n_distinct_r"] = len(np.unique(r))
        d["change_rate"] = np.sum(np.diff(r) != 0) / (n - 1) if n > 1 else 0
        d["concentration"] = np.sum(vc.values ** 2)
        # NEW: rating transition features
        if n > 1:
            diffs = np.diff(r)
            d["mean_abs_diff"] = np.mean(np.abs(diffs))
            d["std_diff"] = np.std(diffs)
            d["p_same_rating"] = np.mean(diffs == 0)
            d["max_run"] = max(len(list(gi)) for _, gi in itertools.groupby(r))
        else:
            d["mean_abs_diff"] = 0; d["std_diff"] = 0
            d["p_same_rating"] = 0; d["max_run"] = n
        # NEW: percentile spread
        if n >= 10:
            d["p10"] = np.percentile(r, 10)
            d["p90"] = np.percentile(r, 90)
            d["p90_p10"] = d["p90"] - d["p10"]
        else:
            d["p10"] = 0; d["p90"] = 0; d["p90_p10"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(dist_f))

    # ===== C. Interaction Structure (expanded) =====
    inter = df.groupby("user").agg(n_unique_items=("item", "nunique"))
    inter["repeat_ratio"] = 1 - inter["n_unique_items"] / df.groupby("user").size()
    inter["density"] = df.groupby("user").size() / n_items
    # NEW: per-item repeat statistics
    ic = df.groupby(["user", "item"]).size().reset_index(name="cnt")
    rs2 = ic.groupby("user")["cnt"].agg(
        max_item_repeat="max", mean_item_repeat="mean"
    ).fillna(0)
    inter = inter.join(rs2, how="left").fillna(0)
    feats.append(inter)

    # ===== D. Item-Normalized Deviation (expanded) =====
    m = df.merge(g_item_stats, left_on="item", right_index=True, how="left")
    m["item_mean"] = m["item_mean"].fillna(g_mean)
    m["item_std"] = m["item_std"].fillna(1)
    m["resid"] = m["rating"] - m["item_mean"]
    m["z_resid"] = np.where(m["item_std"] > 0, m["resid"] / m["item_std"], 0)
    m["abs_resid"] = np.abs(m["resid"])

    r_agg = m.groupby("user").agg(
        mean_resid=("resid", "mean"), std_resid=("resid", "std"),
        abs_mean_resid=("abs_resid", "mean"), max_abs_resid=("abs_resid", "max"),
        med_resid=("resid", "median"),
        mean_z=("z_resid", "mean"), std_z=("z_resid", "std"),
        abs_mean_z=("z_resid", lambda x: np.abs(x).mean()),
        resid_q90=("abs_resid", lambda x: np.percentile(x, 90)),
        n_large_dev=("abs_resid", lambda x: (x > 2).sum()),
    ).fillna(0)
    r_agg["large_dev_ratio"] = r_agg["n_large_dev"] / df.groupby("user").size()
    # NEW: deviation asymmetry (do they deviate up or down?)
    n_pos = m.groupby("user")["resid"].apply(lambda x: (x > 1).sum())
    n_neg = m.groupby("user")["resid"].apply(lambda x: (x < -1).sum())
    r_agg["dev_asymmetry"] = (n_pos - n_neg) / (n_pos + n_neg + 1)
    feats.append(r_agg)

    # ===== E. Popularity-Aware (expanded) =====
    mp = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp["ipop"] = mp["ipop"].fillna(1)
    mp["iw"] = 1.0 / (mp["ipop"] + 1)
    mp["wr"] = mp["rating"] * mp["iw"]
    pf = mp.groupby("user").agg(
        wr_sum=("wr", "sum"), iw_sum=("iw", "sum"),
        mean_ipop=("ipop", "mean"), std_ipop=("ipop", "std"),
        mean_lp=("ipop", lambda x: np.log1p(x).mean()),
    )
    pf["w_mean_r"] = pf["wr_sum"] / pf["iw_sum"]
    pf = pf.drop(columns=["wr_sum", "iw_sum"]).fillna(0)
    # NEW: mean rating by item-popularity quartile
    mp["pop_q"] = pd.qcut(mp["ipop"].rank(method="first"), q=4, labels=False)
    for q in range(4):
        pf[f"mean_r_popq{q}"] = mp[mp["pop_q"] == q].groupby("user")["rating"].mean()
    pf = pf.fillna(0)
    feats.append(pf)

    # ===== F. Sequence Patterns (expanded) =====
    def seq_f(g):
        r = g["rating"].values; n = len(r)
        d = {}
        if n >= 3:
            mid = n // 2
            d["drift"] = np.mean(r[mid:]) - np.mean(r[:mid])
            d["trend"] = np.polyfit(range(n), r, 1)[0]
            d["autocorr"] = np.corrcoef(r[:-1], r[1:])[0, 1] if np.std(r) > 0 else 0
            if np.isnan(d["autocorr"]): d["autocorr"] = 0
        else:
            d["drift"] = 0; d["trend"] = 0; d["autocorr"] = 0
        # NEW: quarter-segment features
        if n >= 10:
            q1, q2, q3, q4 = np.array_split(r, 4)
            d["q4_q1_diff"] = np.mean(q4) - np.mean(q1)
            d["segment_std"] = np.std([np.mean(q1), np.mean(q2), np.mean(q3), np.mean(q4)])
        else:
            d["q4_q1_diff"] = 0; d["segment_std"] = 0
        return pd.Series(d)
    feats.append(df.groupby("user").apply(seq_f))

    # ===== G. Item Diversity =====
    def div_f(g):
        vc = pd.Series(g["item"].values).value_counts(normalize=True)
        return pd.Series({
            "item_entropy": stats.entropy(vc),
            "gini_simpson": 1 - np.sum(vc ** 2),
            "unique_ratio": len(np.unique(g["item"].values)) / n_items,
        })
    feats.append(df.groupby("user").apply(div_f))

    # ===== H. SVD Latent Embeddings (35 components) =====
    uid_map = {u: i for i, u in enumerate(users_list)}
    rows = df["user"].map(uid_map).values
    cols = df["item"].values
    vals = df["rating"].values.astype(float)
    um = csr_matrix((vals, (rows, cols)), shape=(len(users_list), n_items))
    nc_svd = 35  # increased from 30 — more data supports more components
    if fit or svd_model is None:
        svd_model = TruncatedSVD(n_components=nc_svd, random_state=42)
        emb = svd_model.fit_transform(um)
    else:
        emb = svd_model.transform(um)
    svd_df = pd.DataFrame(emb, index=users_list, columns=[f"svd_{i}" for i in range(nc_svd)])
    svd_df.index.name = "user"
    recon = svd_model.inverse_transform(emb)
    svd_df["svd_err"] = np.mean((um.toarray() - recon) ** 2, axis=1)
    svd_df["svd_norm"] = np.linalg.norm(emb, axis=1)
    feats.append(svd_df)

    # ===== H2. NMF Embeddings (NEW — complementary to SVD) =====
    nc_nmf = 15
    um_pos = um.copy()
    um_pos.data = np.clip(um_pos.data, 0, None)  # NMF requires non-negative
    if fit or nmf_model is None:
        nmf_model = NMF(n_components=nc_nmf, random_state=42, max_iter=300, init='nndsvda')
        nmf_emb = nmf_model.fit_transform(um_pos)
    else:
        nmf_emb = nmf_model.transform(um_pos)
    nmf_df = pd.DataFrame(nmf_emb, index=users_list, columns=[f"nmf_{i}" for i in range(nc_nmf)])
    nmf_df.index.name = "user"
    nmf_recon = nmf_model.inverse_transform(nmf_emb)
    nmf_df["nmf_err"] = np.mean((um_pos.toarray() - nmf_recon) ** 2, axis=1)
    nmf_df["nmf_norm"] = np.linalg.norm(nmf_emb, axis=1)
    feats.append(nmf_df)

    # ===== I. Cosine Similarity to Average User =====
    bm = csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(users_list), n_items))
    avg = bm.mean(axis=0).A1
    sims = []
    for i in range(len(users_list)):
        uv = bm[i].toarray().flatten()
        d_val = np.dot(uv, avg); nu = np.linalg.norm(uv); na = np.linalg.norm(avg)
        sims.append(d_val / (nu * na) if nu > 0 and na > 0 else 0)
    feats.append(pd.DataFrame({"cos_sim_avg": sims}, index=users_list).rename_axis("user"))

    # ===== J. NEW: Rating vs Item Popularity Correlation =====
    mp2 = df.merge(g_item_pop.rename("ipop").to_frame(), left_on="item", right_index=True, how="left")
    mp2["ipop"] = mp2["ipop"].fillna(1)
    def pcorr(g):
        if len(g) < 5: return pd.Series({"rating_pop_corr": 0})
        c = np.corrcoef(g["rating"].values, g["ipop"].values)[0, 1]
        return pd.Series({"rating_pop_corr": c if not np.isnan(c) else 0})
    feats.append(mp2.groupby("user").apply(pcorr))

    # ===== K. NEW: Item Overlap Between Halves =====
    def burst_f(g):
        items = g["item"].values; n = len(items)
        if n >= 10:
            h1 = set(items[:n//2]); h2 = set(items[n//2:])
            return pd.Series({"item_overlap_halves": len(h1 & h2) / max(len(h1 | h2), 1)})
        return pd.Series({"item_overlap_halves": 0})
    feats.append(df.groupby("user").apply(burst_f))

    # ===== Combine All =====
    result = feats[0]
    for f in feats[1:]: result = result.join(f, how="left")
    result = result.fillna(0).replace([np.inf, -np.inf], 0)
    return result, svd_model, nmf_model

# Compute features
print("Computing training features...")
X_train_f, svd_m, nmf_m = compute_features(train_ratings, fit=True)
print("Computing test features...")
X_test_f, _, _ = compute_features(test_ratings, svd_model=svd_m, nmf_model=nmf_m, fit=False)
y_train = labels.loc[X_train_f.index, "label"]

print(f"\nTraining features: {X_train_f.shape}")
print(f"Test features:     {X_test_f.shape}")

## 3. Unsupervised Anomaly Scores as Additional Features

In [ ]:
sc_u = RobustScaler()
Xts = sc_u.fit_transform(X_train_f)
Xes = sc_u.transform(X_test_f)

# Multiple Isolation Forest configs for diversity
for i, (cont, mf, rs) in enumerate([(0.085, 0.8, 42), (0.095, 0.6, 77)]):
    iso = IsolationForest(n_estimators=300, contamination=cont, random_state=rs, max_features=mf)
    iso.fit(Xts)
    X_train_f[f"iso_score_{i}"] = -iso.score_samples(Xts)
    X_test_f[f"iso_score_{i}"] = -iso.score_samples(Xes)

lof = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination=0.09)
lof.fit(Xts)
X_train_f["lof_score"] = -lof.score_samples(Xts)
X_test_f["lof_score"] = -lof.score_samples(Xes)

print(f"Final feature count: {X_train_f.shape[1]}")

## 4. Model Training with Multi-Seed 10-Fold Stratified CV

**11 base models × 2 seeds = 22 model variants**:
- **LightGBM ×4**: depth 3–6, varied regularization
- **XGBoost ×3**: complementary gradient boosting configs
- **Random Forest**: max_depth 8, balanced class weights
- **ExtraTrees** (NEW): random split points for added diversity
- **Gradient Boosting**: sklearn implementation
- **Logistic Regression**: regularized linear baseline

Multi-seed training averages OOF predictions across 2 independent CV splits,
reducing variance and producing more reliable ensemble selection.


In [ ]:
fcols = X_train_f.columns.tolist()
X = X_train_f[fcols].values
Xt = X_test_f[fcols].values
y = y_train.values

sc2 = RobustScaler()
Xs = sc2.fit_transform(X)
Xts2 = sc2.transform(Xt)

spw = (len(y) - y.sum()) / y.sum()
print(f"Class imbalance: {spw:.0f}:1 (normal:anomalous)")

def get_models(seed=0):
    return {
        f"lgbm_a_{seed}": lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.025, max_depth=5, num_leaves=20,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42+seed, verbose=-1),
        f"lgbm_b_{seed}": lgb.LGBMClassifier(
            n_estimators=400, learning_rate=0.035, max_depth=4, num_leaves=12,
            min_child_samples=15, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77+seed, verbose=-1),
        f"lgbm_c_{seed}": lgb.LGBMClassifier(
            n_estimators=450, learning_rate=0.03, max_depth=6, num_leaves=25,
            min_child_samples=8, subsample=0.85, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=1.0, scale_pos_weight=spw,
            random_state=123+seed, verbose=-1),
        f"lgbm_d_{seed}": lgb.LGBMClassifier(
            n_estimators=350, learning_rate=0.04, max_depth=3, num_leaves=8,
            min_child_samples=20, subsample=0.7, colsample_bytree=0.5,
            reg_alpha=2.0, reg_lambda=3.0, scale_pos_weight=spw,
            random_state=200+seed, verbose=-1),
        f"xgb_a_{seed}": xgb.XGBClassifier(
            n_estimators=500, learning_rate=0.025, max_depth=5,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.65,
            reg_alpha=0.8, reg_lambda=1.5, scale_pos_weight=spw,
            random_state=42+seed, eval_metric="logloss", verbosity=0),
        f"xgb_b_{seed}": xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.035, max_depth=4,
            min_child_weight=8, subsample=0.75, colsample_bytree=0.55,
            reg_alpha=1.5, reg_lambda=2.5, scale_pos_weight=spw,
            random_state=77+seed, eval_metric="logloss", verbosity=0),
        f"xgb_c_{seed}": xgb.XGBClassifier(
            n_estimators=450, learning_rate=0.03, max_depth=6,
            min_child_weight=3, subsample=0.85, colsample_bytree=0.7,
            reg_alpha=0.5, reg_lambda=1.0, scale_pos_weight=spw,
            random_state=123+seed, eval_metric="logloss", verbosity=0),
        f"rf_{seed}": RandomForestClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"et_{seed}": ExtraTreesClassifier(
            n_estimators=500, max_depth=8, min_samples_leaf=4,
            class_weight="balanced", random_state=42+seed, n_jobs=-1),
        f"gb_{seed}": GradientBoostingClassifier(
            n_estimators=350, learning_rate=0.035, max_depth=4,
            min_samples_leaf=8, subsample=0.8, random_state=42+seed),
        f"lr_{seed}": LogisticRegression(
            C=0.5, class_weight="balanced", max_iter=3000, random_state=42+seed),
    }

print(f"Models per seed: {list(get_models(0).keys())}")
print(f"Total model variants: {len(get_models(0))} × 2 seeds = {len(get_models(0))*2}")

In [ ]:
N_FOLDS = 10
all_oof = {}
all_test_p = {}

for seed in [0, 1000]:
    models_dict = get_models(seed)
    model_names = list(models_dict.keys())
    oof = {n: np.zeros(len(y)) for n in model_names}
    test_p = {n: np.zeros(len(Xt)) for n in model_names}

    cv_seed = 42 + seed
    print(f"\n{'='*60}")
    print(f"Seed={seed}: {N_FOLDS}-fold stratified CV ({len(model_names)} models)")
    print(f"(~{int(260/N_FOLDS)} anomalous users per fold)")
    print(f"{'='*60}")
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=cv_seed)

    for fi, (tidx, vidx) in enumerate(skf.split(X, y)):
        Xtr, Xv = X[tidx], X[vidx]
        Xtr_s, Xv_s = Xs[tidx], Xs[vidx]
        ytr, yv = y[tidx], y[vidx]

        fold_models = get_models(seed)
        fold_info = []
        for name in model_names:
            model = fold_models[name]
            if "lr" in name:
                model.fit(Xtr_s, ytr)
                oof[name][vidx] = model.predict_proba(Xv_s)[:, 1]
                test_p[name] += model.predict_proba(Xts2)[:, 1] / N_FOLDS
            else:
                model.fit(Xtr, ytr)
                oof[name][vidx] = model.predict_proba(Xv)[:, 1]
                test_p[name] += model.predict_proba(Xt)[:, 1] / N_FOLDS

            bf = max(f1_score(yv, (oof[name][vidx] >= t).astype(int))
                     for t in np.arange(0.05, 0.95, 0.01))
            fold_info.append(f"{name.split('_' + str(seed))[0]}:{bf:.3f}")

        print(f"  Fold {fi+1}: {', '.join(fold_info)}")

    all_oof.update(oof)
    all_test_p.update(test_p)

    print(f"\nSeed {seed} OOF Performance:")
    for n in model_names:
        auc = roc_auc_score(y, oof[n])
        bf = max(f1_score(y, (oof[n] >= t).astype(int)) for t in np.arange(0.01, 0.99, 0.005))
        print(f"  {n:17s}: AUC={auc:.4f}, F1={bf:.4f}")

## 5. Ensemble Selection & Threshold Optimization

Evaluates:
- **Individual models** — best single model
- **Multi-seed averaging** — same architecture averaged across seeds (variance reduction)
- **Top-N equal-weight averaging** (N = 5, 8, 10, 15, 20)
- **AUC³-weighted averaging** — better models weighted exponentially
- **Rank-based averaging** — robust to different score scales
- **Stacking** — LR meta-learner on OOF predictions + ranks (multi-C)


In [ ]:
def best_f1_thr(scores, y_true):
    """Find threshold maximizing F1."""
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y_true, (scores >= t).astype(int))
        if f > bf: bf, bt = f, t
    return bf, bt

all_model_names = list(all_oof.keys())
model_aucs = {n: roc_auc_score(y, all_oof[n]) for n in all_model_names}
sorted_m = sorted(model_aucs, key=model_aucs.get, reverse=True)
results = {}

# --- Individual models (top 10) ---
for n in sorted_m[:10]:
    f1, thr = best_f1_thr(all_oof[n], y)
    results[f"solo_{n[:14]}"] = (f1, thr, {n: 1.0})

# --- Multi-seed averaging (same architecture across seeds) ---
base_names = set()
for n in all_model_names:
    # Strip seed suffix: "lgbm_a_0" -> "lgbm_a", "lgbm_a_1000" -> "lgbm_a"
    for sfx in ["_0", "_1000"]:
        if n.endswith(sfx):
            base_names.add(n[:-len(sfx)])

for bn in sorted(base_names):
    matching = [n for n in all_model_names if any(n == f"{bn}_{s}" for s in [0, 1000])]
    if len(matching) >= 2:
        avg_oof = np.mean([all_oof[n] for n in matching], axis=0)
        avg_test = np.mean([all_test_p[n] for n in matching], axis=0)
        f1, thr = best_f1_thr(avg_oof, y)
        results[f"mseed_{bn}"] = (f1, thr, {"__custom__": True, "__oof__": avg_oof, "__test__": avg_test})

# --- Top-N ensembles by AUC ---
for top_n in [5, 8, 10, 15, 20]:
    top = sorted_m[:top_n]
    # Equal weights
    oof_e = np.mean([all_oof[n] for n in top], axis=0)
    test_e = np.mean([all_test_p[n] for n in top], axis=0)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_eq"] = (f1, thr, {"__custom__": True, "__oof__": oof_e, "__test__": test_e})
    # AUC^3 weights
    ws = {n: model_aucs[n]**3 for n in top}; sw = sum(ws.values())
    ws_norm = {n: w/sw for n, w in ws.items()}
    oof_e = sum(ws_norm[n] * all_oof[n] for n in top)
    test_e = sum(ws_norm[n] * all_test_p[n] for n in top)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_auc3"] = (f1, thr, {"__custom__": True, "__oof__": oof_e, "__test__": test_e})
    # Rank-based
    oof_ranks = {n: rankdata(all_oof[n]) / len(y) for n in top}
    oof_e = np.mean([oof_ranks[n] for n in top], axis=0)
    test_ranks = {n: rankdata(all_test_p[n]) / len(Xt) for n in top}
    test_e = np.mean([test_ranks[n] for n in top], axis=0)
    f1, thr = best_f1_thr(oof_e, y)
    results[f"top{top_n}_rank"] = (f1, thr, {"__custom__": True, "__oof__": oof_e, "__test__": test_e})

# --- Stacking with LR meta-learner (multi-C) ---
for top_n in [8, 10, 15]:
    top = sorted_m[:top_n]
    s_X = np.column_stack([all_oof[n] for n in top])
    s_Xt = np.column_stack([all_test_p[n] for n in top])
    # Add rank features for robustness
    s_X = np.hstack([s_X, np.column_stack([rankdata(all_oof[n])/len(y) for n in top])])
    s_Xt = np.hstack([s_Xt, np.column_stack([rankdata(all_test_p[n])/len(Xt) for n in top])])

    m_oof = np.zeros(len(y)); m_test = np.zeros(len(Xt))
    n_C = 3
    skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for ti, vi in skf2.split(s_X, y):
        for C in [0.1, 0.5, 1.0]:
            mm = LogisticRegression(C=C, class_weight="balanced", max_iter=3000, random_state=42)
            mm.fit(s_X[ti], y[ti])
            m_oof[vi] += mm.predict_proba(s_X[vi])[:, 1] / n_C
            m_test += mm.predict_proba(s_Xt)[:, 1] / (5 * n_C)
    f1, thr = best_f1_thr(m_oof, y)
    results[f"stack_top{top_n}"] = (f1, thr, {"__custom__": True, "__oof__": m_oof, "__test__": m_test})

# Print rankings
print("Ensemble Configuration Rankings (by OOF F1):")
print("=" * 60)
sorted_results = sorted(results.items(), key=lambda x: -x[1][0])
for i, (name, (f1, thr, ws)) in enumerate(sorted_results[:25]):
    marker = " <<<" if i == 0 else ""
    print(f"  {i+1:2d}. {name:22s}: F1={f1:.4f} @ thr={thr:.3f}{marker}")

best_name = max(results, key=lambda x: results[x][0])
best_f1, best_thr, best_ws = results[best_name]
print(f"\n>>> Selected: {best_name} (F1={best_f1:.4f}, threshold={best_thr:.3f})")

In [ ]:
# Generate final predictions using the best configuration
if "__custom__" in best_ws:
    final_oof = best_ws["__oof__"]
    final_test = best_ws["__test__"]
else:
    ws = {k: v for k, v in best_ws.items() if not k.startswith("__")}
    final_oof = sum(ws[n] * all_oof[n] for n in ws)
    final_test = sum(ws[n] * all_test_p[n] for n in ws)

final_thr = best_thr

## 6. Threshold Robustness Validation

In [ ]:
thr_folds = []
skf3 = StratifiedKFold(n_splits=10, shuffle=True, random_state=77)
for ti, vi in skf3.split(np.zeros(len(y)), y):
    bf, bt = 0, 0.5
    for t in np.arange(0.01, 0.99, 0.005):
        f = f1_score(y[vi], (final_oof[vi] >= t).astype(int))
        if f > bf: bf, bt = f, t
    thr_folds.append(bt)

print("Threshold Robustness Analysis:")
print(f"  Per-fold thresholds: {[f'{t:.3f}' for t in thr_folds]}")
print(f"  Mean:   {np.mean(thr_folds):.3f}")
print(f"  Median: {np.median(thr_folds):.3f}")
print(f"  Std:    {np.std(thr_folds):.3f}")
print(f"  Global optimal: {final_thr:.3f}")

In [ ]:
preds = (final_oof >= final_thr).astype(int)

print("=" * 60)
print("FINAL OUT-OF-FOLD PERFORMANCE")
print("=" * 60)
print(f"  AUC:       {roc_auc_score(y, final_oof):.4f}")
print(f"  F1:        {f1_score(y, preds):.4f}")
print(f"  Precision: {precision_score(y, preds):.4f}")
print(f"  Recall:    {recall_score(y, preds):.4f}")
print(f"  Threshold: {final_thr:.3f}")
print()
print(classification_report(y, preds, target_names=["Normal", "Anomalous"]))

## 7. Save Predictions

In [ ]:
test_users = X_test_f.index.values
pred_dict = {int(u): float(s) for u, s in zip(test_users, final_test)}

# Save in required format
np.savez("submission.npz", predictions=final_test)

# Detailed CSV
res_df = pd.DataFrame({
    "user": test_users, "anomaly_score": final_test,
    "predicted_label": (final_test >= final_thr).astype(int)
}).sort_values("anomaly_score", ascending=False)
res_df.to_csv("predictions.csv", index=False)

# JSON dict
with open("predictions.json", "w") as f:
    json.dump({"predictions": pred_dict}, f)

n_anom = (final_test >= final_thr).sum()
print(f"Predictions saved for {len(test_users)} test users")
print(f"Predicted anomalies: {n_anom} ({n_anom/len(test_users)*100:.1f}%)")
print(f"Score stats: mean={final_test.mean():.4f}, std={final_test.std():.4f}")
print(f"\nTop 20 most anomalous users:")
print(res_df.head(20)[["user", "anomaly_score"]].to_string(index=False))

## Summary

### v2 → v3 Improvements
| Aspect | v2 (2 batches) | v3 (3 batches) |
|--------|----------------|----------------|
| Training users | 2200 (200 anom) | **3060 (260 anom)** |
| Features | 86 | **~127** |
| Model variants | 8 | **22 (11 × 2 seeds)** |
| CV folds | 7 | **10** |
| CV seeds | 1 | **2 (variance reduction)** |
| SVD components | 30 | **35** |
| NMF embeddings | — | **15 components** |
| ExtraTrees | — | **Added** |
| Stacking meta-learner | single C | **multi-C averaging** |

### Key changes enabling improvement
- **3× training data** — 260 positives gives much more stable CV estimates
- **NMF embeddings** — complementary decomposition to SVD (parts-based vs spectral)
- **Rating transition features** — capture sequential behavior patterns anomalous users exhibit
- **Popularity-quartile behavior** — how users rate popular vs niche items
- **Multi-seed CV** — reduces OOF variance, more reliable ensemble selection
- **Deviation asymmetry** — whether users systematically over- or under-rate vs item means
